# Paper 1 — Golden Age Semantic Reconfiguration

**Working title:** *Reconfiguring the Golden Age: Semantic Networks and the Renaissance–Baroque Transition in Spanish Poetry*

> **Current stage: Phase 3 — Herrera edition resolution + temporal design.**

Phase 2 established that the TEI corpus does not contain usable poem-level chronology: only one file carries witness/edition dates, and the 5,078 poems collapse to **57 author–source groups**. We therefore now resolve the special Herrera layers and formalize a **two-axis temporal design**: composition history (primary) and publication/circulation history (secondary sensitivity analysis).

## Colab ↔ GitHub workflow

1. Open this notebook from `ardominguezm/golden-age-semantic-reconfiguration`.
2. Run **Runtime → Run all**.
3. Inspect the final checkpoint.
4. Save the executed notebook back to this same GitHub path on `main`.

Do **not** build semantic networks yet. This phase decides what temporal information is defensible.

## What Phase 2 established

- Navarro TEI: **5,078 poems**, 53 author folders.
- The TEI itself supplies effectively **no composition chronology**.
- Only 1 TEI file has witness/edition dates; none has a separate composition date.
- The corpus reduces to **57 author + source-description groups**, so external dating can be organized by source/collection rather than by 5,078 isolated searches.
- `AN` and `Herrera` in the Hernández network corpus are not duplicate layers: they share only **one exact normalized poem**.
- Pacheco contributes **22 source-exclusive sonnets** absent from Navarro.
- The previous literature identifies the unique posthumous Herrera poems as **P2**, drawn from *Versos de Fernando de Herrera* (1619), while **H** denotes *Algunas obras* (1582).

In [ ]:
import sys, re, shutil, subprocess, unicodedata
from pathlib import Path
from collections import defaultdict
import pandas as pd
import xml.etree.ElementTree as ET

SOURCES = {
    "network": (
        "https://github.com/lamusadecima/Network_for_Golden_Age_Spanish_Poetry.git",
        "ef6b7b691f67abe60d9cfa85c274f0be8095dd9a",
    ),
    "navarro": (
        "https://github.com/bncolorado/CorpusSonetosSigloDeOro.git",
        "092a5fe70a4065a4d84bfed288bffd3851348f9c",
    ),
    "digital_stylistics": (
        "https://github.com/lamusadecima/Digital-Stylistics-Applied-to-Golden-Age.git",
        "0de990eac908897b5e931aeb5c496170ccf35bab",
    ),
}

ROOT = Path("/content/gasr_sources")
ROOT.mkdir(exist_ok=True)

def clone(name, url, commit):
    dst = ROOT / name
    if dst.exists():
        shutil.rmtree(dst)
    subprocess.run(["git","clone","--quiet",url,str(dst)], check=True)
    subprocess.run(["git","-C",str(dst),"checkout","--quiet",commit], check=True)
    got = subprocess.check_output(["git","-C",str(dst),"rev-parse","HEAD"], text=True).strip()
    assert got == commit, (name, got, commit)
    return dst

paths = {k: clone(k, *v) for k,v in SOURCES.items()}
NET, NAV, DST = paths["network"], paths["navarro"], paths["digital_stylistics"]
NS = {"tei":"http://www.tei-c.org/ns/1.0"}

def norm(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]","",s.lower())

def blocks(text):
    out, cur = [], []
    for line in text.splitlines():
        if line.strip():
            cur.append(line.strip())
        elif cur:
            out.append(cur); cur=[]
    if cur: out.append(cur)
    return out

def element_text(el):
    return "" if el is None else " ".join(" ".join(el.itertext()).split())

print("Pinned sources ready")
for k,v in paths.items():
    print(" ",k,":",v)
print("Python",sys.version.split()[0],"| pandas",pd.__version__)

## 01. Identify `AN` empirically as P2

The network repository contains `AN_SonetosP2.txt`; the companion *Digital Stylistics Applied to Golden Age* repository contains explicit `H.txt` and `P2.txt`.

Hernández-Lorenzo defines:

- **H** = sonnets from *Algunas obras* (1582), published by Herrera during his lifetime.
- **P2** = the **unique sonnets** in the posthumous *Versos de Fernando de Herrera* (1619), excluding poems already published in H.

We test this identity from the texts rather than relying only on the filename.

In [ ]:
def file_blocks(path, label):
    rows=[]
    for i,b in enumerate(blocks(path.read_text(encoding="utf-8",errors="replace")),1):
        t="\n".join(b)
        rows.append({"layer":label,"local_id":i,"n_lines":len(b),"text":t,"signature":norm(t)})
    return pd.DataFrame(rows)

net_AN = file_blocks(NET/"corpus"/"AN_SonetosP2.txt", "network_AN")
net_Herrera = file_blocks(NET/"corpus"/"Herrera_Sonetos.txt", "network_Herrera")
net_Pacheco = file_blocks(NET/"corpus"/"Pacheco_Sonetos.txt", "network_Pacheco")

U = DST/"corpus"/"untagged_corpus"
dst_H = file_blocks(U/"H.txt", "digital_H_1582")
dst_P2 = file_blocks(U/"P2.txt", "digital_P2_1619")

layers = pd.DataFrame([
    {"layer":"network_AN","blocks":len(net_AN),"blocks_14":int((net_AN.n_lines==14).sum())},
    {"layer":"network_Herrera","blocks":len(net_Herrera),"blocks_14":int((net_Herrera.n_lines==14).sum())},
    {"layer":"network_Pacheco","blocks":len(net_Pacheco),"blocks_14":int((net_Pacheco.n_lines==14).sum())},
    {"layer":"digital_H_1582","blocks":len(dst_H),"blocks_14":int((dst_H.n_lines==14).sum())},
    {"layer":"digital_P2_1619","blocks":len(dst_P2),"blocks_14":int((dst_P2.n_lines==14).sum())},
])
display(layers)

def overlap(a,b):
    A,B=set(a.signature),set(b.signature)
    return {
        "A":a.layer.iloc[0], "B":b.layer.iloc[0],
        "unique_A":len(A), "unique_B":len(B),
        "exact_shared":len(A&B),
        "coverage_A":len(A&B)/len(A) if A else 0,
        "coverage_B":len(A&B)/len(B) if B else 0,
    }

edition_overlap = pd.DataFrame([
    overlap(net_AN,dst_P2),
    overlap(net_AN,dst_H),
    overlap(net_Herrera,dst_H),
    overlap(net_Herrera,dst_P2),
    overlap(dst_H,dst_P2),
])
display(edition_overlap)

### Interpretation rule

If `network_AN` has overwhelming exact overlap with `digital_P2_1619` and little overlap with H, we relabel it **Herrera_P2** in the master metadata.

Likewise, exact overlap between `network_Herrera` and `digital_H_1582` identifies a subset with secure **publication/circulation year 1582**. Remaining `network_Herrera` poems are not automatically assigned 1582 because the network study explicitly notes that the file also contains dispersed undoubted poems.

## 02. Recover Navarro Herrera identities and source structure

Navarro remains the poem-identity backbone. We now ask which Navarro Herrera poems can be securely tagged as H or P2 by exact normalized text.

In [ ]:
nrows=[]
for p in sorted(NAV.rglob("*.xml")):
    root=ET.parse(p).getroot()
    lines=[element_text(x) for x in root.findall(".//tei:l",NS)]
    lines=[x for x in lines if x]
    bibl=element_text(root.find(".//tei:sourceDesc/tei:bibl",NS))
    text="\n".join(lines)
    nrows.append({
        "n_id":str(p.relative_to(NAV)).replace("/","::"),
        "author_dir":p.parent.name,
        "title":element_text(root.find(".//tei:body/tei:head/tei:title",NS)),
        "text_tei":text,
        "n_lines":len(lines),
        "source_bibl":bibl,
        "signature":norm(text),
    })
 n=pd.DataFrame(nrows)

herr=n[n.author_dir.eq("FernandoDeHerrera")].copy()
Hsig=set(dst_H.signature)
P2sig=set(dst_P2.signature)
herr["edition_exact"]=herr.signature.map(
    lambda s: "H_1582" if s in Hsig else ("P2_1619" if s in P2sig else "unclassified")
)
print("Navarro Herrera poems:",len(herr))
display(herr.edition_exact.value_counts().rename_axis("edition_exact").reset_index(name="poems"))
display(herr[["n_id","title","edition_exact","source_bibl"]].head(40))

## 03. Build temporal evidence channels — never collapse them

For the paper we keep two distinct historical axes:

**A. Composition axis — primary.**  
This is the axis relevant to the question “when was the semantic system of poetry being reorganized?” It must use scholarly composition dates or bounded composition intervals. Publication year is not silently substituted for composition year.

**B. Publication/circulation axis — secondary sensitivity analysis.**  
This asks when a textual configuration entered print circulation. It is particularly informative for Herrera because P2 was published posthumously in 1619. Treating 1619 as a composition date would be a serious historical error; treating it as a circulation date is legitimate.

The difference between these two axes can itself become an interpretive robustness test.

In [ ]:
temporal_schema = pd.DataFrame([
    ["composition_exact","composition", "A", "exact/near-exact scholarly composition year", "primary"],
    ["composition_interval","composition", "A/B", "bounded scholarly composition interval", "primary"],
    ["first_publication","circulation", "B", "first publication/collection year", "secondary"],
    ["witness_or_edition","bibliographic", "C", "date of witness/edition only", "never composition"],
    ["author_activity_interval","composition", "D", "fallback bounded author activity interval", "uncertainty only"],
    ["author_lifespan","biographical", "E", "biographical bound", "covariate / last-resort bound"],
], columns=["date_type","axis","confidence_tier","meaning","role"])
display(temporal_schema)

herr_temporal = herr[["n_id","title","edition_exact"]].copy()
herr_temporal["circulation_year"] = herr_temporal.edition_exact.map({"H_1582":1582,"P2_1619":1619})
herr_temporal["composition_date_min"] = pd.NA
herr_temporal["composition_date_max"] = pd.NA
herr_temporal["composition_status"] = "requires scholarly dating; do not use circulation year as composition"
herr_temporal["circulation_status"] = herr_temporal.edition_exact.map({
    "H_1582":"secure edition membership if exact text match",
    "P2_1619":"secure first-print circulation if exact P2 match",
    "unclassified":"not assigned from exact edition evidence",
})
display(herr_temporal.head(30))

## 04. External dating worklist by author + source

Phase 2 showed that the entire Navarro corpus reduces to only 57 author–source-description groups. We reproduce that worklist here and make the next scholarly task explicit: locate a reliable critical chronology for each high-priority group.

This avoids the infeasible and methodologically noisy strategy of searching 5,078 poems independently.

In [ ]:
source_groups=(n.groupby(["author_dir","source_bibl"],dropna=False)
    .agg(poems=("n_id","count"))
    .reset_index()
    .sort_values(["poems","author_dir"],ascending=[False,True]))

central = {
    "GarcilasoDeLaVega","JuanBoscan","FernandoDeHerrera","PedroEspinosa",
    "JuanDeArguijo","JuanDeJauregui","LuisCarrilloYSotomayor","Cervantes",
    "Gongora","LopeDeVega_1","LopeDeVega_2","Quevedo"
}
source_groups["research_priority"] = source_groups.apply(
    lambda r:"A — central transition author"
    if r.author_dir in central else ("B — high corpus weight" if r.poems>=100 else "C — standard"),
    axis=1
)
source_groups["composition_evidence_needed"] = (
    "critical edition / scholarly chronology: exact year or bounded composition interval"
)
source_groups["fallback_if_unavailable"] = (
    "first publication/circulation year + author activity bounds; retain as interval uncertainty"
)

order={"A — central transition author":0,"B — high corpus weight":1,"C — standard":2}
source_groups["_o"]=source_groups.research_priority.map(order)
source_groups=source_groups.sort_values(["_o","poems"],ascending=[True,False]).drop(columns="_o")

print("Author + source-description groups:",len(source_groups))
print("Authors:",n.author_dir.nunique())
display(source_groups.head(35))

## 05. Decision rule after this run

After saving the executed notebook, we will decide:

1. whether `AN` can be definitively renamed **P2** from text identity;
2. how many Herrera poems obtain secure H/P2 edition membership;
3. whether 1582/1619 are usable only on the circulation axis (expected) or provide any additional composition bounds;
4. which 57 source groups need external scholarly chronology first;
5. whether a composition-time dynamic network has enough dated/interval-dated material to proceed, or whether the main temporal model must explicitly integrate interval uncertainty.

No temporal windows are selected in this phase.

In [ ]:
OUT=Path("/content/gasr_phase3_outputs"); OUT.mkdir(exist_ok=True)
edition_overlap.to_csv(OUT/"herrera_edition_overlap.csv",index=False)
herr_temporal.to_csv(OUT/"herrera_temporal_evidence.csv",index=False)
source_groups.to_csv(OUT/"temporal_source_worklist.csv",index=False)
temporal_schema.to_csv(OUT/"temporal_schema.csv",index=False)

print("PHASE 3 CHECKPOINT")
print("------------------")
print("Save this executed notebook to GitHub.")
print("Do NOT build semantic networks yet.")
print("Next: interpret H/P2 identity and begin external scholarly chronology for priority-A sources.")
print("\nRuntime exports:")
for p in sorted(OUT.glob("*.csv")):
    print(" ",p.name)